In [1]:
from tree import Mbtree
from tqdm import tqdm
import gzip, pickle

def calc_and_save_bestmoves_and_score_by_board(self, path):
    bestmoves_and_score_by_board = {}
    for node in tqdm(self.nodelist):
        txt = node.mb.board_to_str()
        if not txt in bestmoves_and_score_by_board.keys():
            bestmoves_and_score_by_board[txt] = {
                "bestmoves": [node.mb.board.move_to_xy(move) for move in node.bestmoves],
                "score": node.score,
                "playout_prob": { 
                    node.mb.CIRCLE_STR: node.playout_prob[node.mb.CIRCLE],
                    node.mb.CROSS_STR: node.playout_prob[node.mb.CROSS],
                    node.mb.DRAW: node.playout_prob[node.mb.DRAW],
                }
            }

    with gzip.open(path, "wb") as f:
        pickle.dump(bestmoves_and_score_by_board, f)
    
    return bestmoves_and_score_by_board  

Mbtree.calc_and_save_bestmoves_and_score_by_board = calc_and_save_bestmoves_and_score_by_board

In [2]:
mbtree = Mbtree()
mbtree.calc_and_save_bestmoves_and_score_by_board("../data/bestmoves_and_score_by_board.dat")
mbtree = Mbtree(shortest_victory=True)
mbtree.calc_and_save_bestmoves_and_score_by_board("../data/bestmoves_and_score_by_board_shortest_victory.dat")
mbtree = Mbtree(shortest_victory=True, recalculate_draw_score=True)
mbtree.calc_and_save_bestmoves_and_score_by_board("../data/bestmoves_and_score_by_board_sv_rd.dat")

     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946


100%|██████████| 549946/549946 [00:02<00:00, 204998.32it/s]


     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946


100%|██████████| 549946/549946 [00:03<00:00, 161437.39it/s]


     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946


100%|██████████| 549946/549946 [00:03<00:00, 151415.58it/s]


{'.........': {'bestmoves': [(0, 0),
   (0, 1),
   (0, 2),
   (1, 0),
   (1, 1),
   (1, 2),
   (2, 0),
   (2, 1),
   (2, 2)],
  'score': -0.5,
  'playout_prob': {'o': 0.5849206349206348,
   'x': 0.2880952380952381,
   'draw': 0.126984126984127}},
 'o........': {'bestmoves': [(1, 1)],
  'score': 0.5,
  'playout_prob': {'o': 0.6071428571428571,
   'x': 0.26428571428571423,
   'draw': 0.1285714285714286}},
 '.o.......': {'bestmoves': [(0, 0), (0, 2), (1, 1), (2, 1)],
  'score': 0.5,
  'playout_prob': {'o': 0.5357142857142857,
   'x': 0.3357142857142857,
   'draw': 0.1285714285714286}},
 '..o......': {'bestmoves': [(1, 1)],
  'score': 0.5,
  'playout_prob': {'o': 0.6071428571428571,
   'x': 0.2642857142857143,
   'draw': 0.1285714285714286}},
 '...o.....': {'bestmoves': [(0, 0), (1, 1), (1, 2), (2, 0)],
  'score': 0.5,
  'playout_prob': {'o': 0.5357142857142857,
   'x': 0.3357142857142857,
   'draw': 0.1285714285714286}},
 '....o....': {'bestmoves': [(0, 0), (0, 2), (2, 0), (2, 2)],
  'sco

In [3]:
from tree import Mbtree_GUI

Mbtree_GUI()

In [4]:
from tree import Node
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_subtree(self, centernode=None, selectednode=None, ax=None, anim_frame=None,
                    isscore=False, show_bestmove=False, show_score=True, size=0.25, lw=0.8, maxdepth=2):   
    def calc_darkness(node):
        if show_bestmove:
            if node.parent is None:
                return 0, show_score
            elif node.mb.board.move_to_xy(node.mb.last_move) in node.parent.bestmoves:
                return 0, show_score
            else:
                return 0.2, show_score
            
        if anim_frame is None:
            return 0, show_score
        pruned_index = getattr(node, "pruned_index", None)
        if pruned_index is not None and anim_frame >= pruned_index:
            return (0.8, False)
        index = node.score_index if isscore else node.id
        return (0.5, False) if index > anim_frame else (0, isscore)
    
    self.nodes_by_rect = {}

    if centernode is None:
        centernode = self.root
    self.calc_node_height(N=centernode, maxdepth=maxdepth)
    if show_bestmove:
        width = 5 * 10
    else:
        width = 5 * (maxdepth + 1)
    height = centernode.height
    parent = centernode.parent
    if parent is not None:
        height += (len(parent.children) - 1) * 4
        parent.height = height
    if ax is None:
        fig, ax = plt.subplots(figsize=(width * size, (height + 1) * size))
        ax.set_xlim(0, width)
        ax.set_ylim(-1, height)   
        ax.invert_yaxis()
        ax.axis("off")        
    
    bottom, top = ax.get_ylim()
    top = -1
    for depth in range(1, 10, 2):
        ax.add_artist(patches.Rectangle(xy=(depth * 5 - 1, top), width=5,
                                        height=bottom - top, fc="whitesmoke"))               

    if show_bestmove:
        bestx = 5 * maxdepth + 4
        bestwidth = 50 - bestx
        ax.add_artist(patches.Rectangle(xy=(bestx, -1), width=bestwidth,
                                        height=height + 1, fc="lightgray"))
    
    nodelist = [centernode]
    depth = centernode.depth
    while len(nodelist) > 0 and depth <= maxdepth:        
        dy = 0
        if parent is not None:
            dy = parent.children.index(centernode) * 4
        childnodelist = []
        for node in nodelist:
            if node is None:
                dy += 4
                childnodelist.append(None)
            else:
                dx = 5 * node.depth
                emphasize = node is selectednode
                darkness, show_score = calc_darkness(node)
                rect = node.draw_node(ax=ax, maxdepth=maxdepth, emphasize=emphasize, darkness=darkness,
                                    show_score=show_score, size=size, lw=lw, dx=dx, dy=dy)
                self.nodes_by_rect[rect] = node
                if show_bestmove and depth == maxdepth:
                    bestnode = node
                    while len(bestnode.bestmoves) > 0:
                        x, y = bestnode.bestmoves[0]
                        bestmove = bestnode.mb.board.xy_to_move(x, y)
                        bestnode = bestnode.children_by_move[bestmove]
                        dx = 5 * bestnode.depth
                        bestnode.height = 4
                        emphasize = bestnode is selectednode
                        rect = bestnode.draw_node(ax=ax, maxdepth=bestnode.depth, emphasize=emphasize,
                                                show_score=show_score, size=size, lw=lw, dx=dx, dy=dy)
                        self.nodes_by_rect[rect] = bestnode                                          
                    
                dy += node.height
                if len(node.children) > 0:  
                    childnodelist += node.children
                else:
                    childnodelist.append(None)
        depth += 1
        nodelist = childnodelist
        
    if parent is not None:
        dy = 0
        for sibling in parent.children:
            if sibling is not centernode:
                sibling.height = 4
                dx = 5 * sibling.depth
                darkness, show_score  = calc_darkness(sibling)
                rect = sibling.draw_node(ax, maxdepth=sibling.depth, size=size, darkness=darkness,
                                        show_score=show_score, lw=lw, dx=dx, dy=dy)
                self.nodes_by_rect[rect] = sibling
            dy += sibling.height
        dx = 5 * parent.depth
        darkness, show_score  = calc_darkness(parent)
        rect = parent.draw_node(ax, maxdepth=maxdepth, darkness=darkness, 
                                show_score=show_score, size=size, lw=lw, dx=dx, dy=0)
        self.nodes_by_rect[rect] = parent
    
        node = parent
        while node.parent is not None:
            node = node.parent
            node.height = height
            dx = 5 * node.depth
            darkness, show_score  = calc_darkness(node)
            rect = node.draw_node(ax, maxdepth=node.depth, darkness=darkness,
                                show_score=show_score, size=size, lw=lw, dx=dx, dy=0)
            self.nodes_by_rect[rect] = node
            
Mbtree.draw_subtree = draw_subtree

In [5]:
Mbtree_GUI()

In [6]:
import ipywidgets as widgets

def create_widgets(self):
    self.output = widgets.Output()  
    self.print_helpmessage()
    self.output.layout.display = "none"
    self.left_button = self.create_button("←", 50)
    self.up_button = self.create_button("↑", 50)
    self.right_button = self.create_button("→", 50)
    self.down_button = self.create_button("↓", 50)
    self.data_dropdown= widgets.Dropdown(
        options = {
            "表示なし": False,
            "評価値": "score",
            "プレイアウトでの〇の勝率": self.selectednode.mb.CIRCLE_STR,
            "プレイアウトでの×の勝率": self.selectednode.mb.CROSS_STR,
            "プレイアウトでの引き分け率": self.selectednode.mb.DRAW,
        },
        description = "データの表示",
        value = "score"
    )
    self.size_slider = widgets.FloatSlider(min=0.05, max=0.25, step=0.01, description="size", value=self.size)
    self.help_button = self.create_button("？", 50)
    self.label = widgets.Label(value="", layout=widgets.Layout(width=f"50px"))
    
    with plt.ioff():
        self.fig = plt.figure(figsize=[self.width * self.size,
                                        self.height * self.size])
        self.ax = self.fig.add_axes([0, 0, 1, 1])
    self.fig.canvas.toolbar_visible = False
    self.fig.canvas.header_visible = False
    self.fig.canvas.footer_visible = False
    self.fig.canvas.resizable = False    
    
    self.dropdown = widgets.Dropdown(
        options=self.scoretable_dict,
        description="score table",
    )
    self.bestmoves_and_score_by_board = self.dropdown.value   
    
Mbtree_GUI.create_widgets = create_widgets

In [7]:
def display_widgets(self):
    hbox1 = widgets.HBox([self.label, self.up_button, self.label, self.label, self.data_dropdown])
    hbox2 = widgets.HBox([self.left_button, self.label, self.right_button,
                        self.size_slider, self.help_button])
    hbox3 = widgets.HBox([self.label, self.down_button, self.label, self.dropdown])
    self.vbox = widgets.VBox([self.output, hbox1, hbox2, hbox3, self.fig.canvas])
    display(self.vbox)  
    
Mbtree_GUI.display_widgets = display_widgets

In [8]:
def create_event_handler(self):
    def on_left_button_clicked(b=None):
        if self.selectednode.parent is not None:
            self.selectednode = self.selectednode.parent
            self.update_gui()
            
    def on_right_button_clicked(b=None):
        if len(self.selectednode.children) > 0:
            self.selectednode = self.selectednode.children[0]
            self.update_gui()

    def on_up_button_clicked(b=None):
        if self.selectednode.parent is not None:
            index = self.selectednode.parent.children.index(self.selectednode)
            if index > 0:
                self.selectednode = self.selectednode.parent.children[index - 1]
                self.update_gui()
            
    def on_down_button_clicked(b=None):
        if self.selectednode.parent is not None:
            index = self.selectednode.parent.children.index(self.selectednode)
            if self.selectednode.parent.children[-1] is not self.selectednode:
                self.selectednode = self.selectednode.parent.children[index + 1]
                self.update_gui()            
                
    def on_data_changed(changed):
        self.show_score = self.data_dropdown.value
        self.update_gui()
                
    def on_size_slider_changed(changed):
        self.size = changed["new"]
        self.fig.set_figwidth(self.width * self.size)
        self.fig.set_figheight(self.height * self.size)
        self.update_gui()
                
    def on_help_button_clicked(b=None):
        self.output.layout.display = "none" if self.output.layout.display is None else None
        self.update_gui()
                        
    self.left_button.on_click(on_left_button_clicked)
    self.right_button.on_click(on_right_button_clicked)
    self.up_button.on_click(on_up_button_clicked)
    self.down_button.on_click(on_down_button_clicked)
    self.data_dropdown.observe(on_data_changed, names="value")
    self.size_slider.observe(on_size_slider_changed, names="value")
    self.help_button.on_click(on_help_button_clicked)

    def on_dropdown_changed(changed):
        self.bestmoves_and_score_by_board = self.dropdown.value
        self.update_gui()

    self.dropdown.observe(on_dropdown_changed, names="value")

    def on_key_press(event):
        keymap = {
            "left": on_left_button_clicked,
            "0": on_left_button_clicked,
            "right": on_right_button_clicked,
            "up": on_up_button_clicked,
            "down": on_down_button_clicked,
        }
        if event.key in keymap:
            keymap[event.key]()
        else:
            try:
                num = int(event.key) - 1
                x = num % 3
                y = 2 - (num // 3)
                move = self.selectednode.mb.board.xy_to_move(x, y)
                if move in self.selectednode.children_by_move:
                    self.selectednode = self.selectednode.children_by_move[move]
                    self.update_gui()
            except:
                pass            
            
    def on_mouse_down(event):
        for rect, node in self.mbtree.nodes_by_rect.items():
            if rect.is_inside(event.xdata, event.ydata):
                self.selectednode = node
                self.update_gui()
                break               
            
    # fig の画像イベントハンドラを結び付ける
    self.fig.canvas.mpl_connect("key_press_event", on_key_press)
    self.fig.canvas.mpl_connect("button_press_event", on_mouse_down)
    
Mbtree_GUI.create_event_handler = create_event_handler

In [9]:
def update_gui(self):
    self.ax.clear()
    self.ax.set_xlim(-1, self.width - 1)
    self.ax.set_ylim(-1, self.height - 1)   
    self.ax.invert_yaxis()
    self.ax.axis("off")   
    
    if self.selectednode.depth <= 4:
        maxdepth = self.selectednode.depth + 1
    elif self.selectednode.depth == 5:
        maxdepth = 7
    else:
        maxdepth = 9
    if self.selectednode.depth <= 6:
        centermb = self.selectednode.mb
    else:
        centermb = Marubatsu()
        for move in self.selectednode.mb.records[1:7]:
            centermb.move(move)
    self.mbtree = Mbtree(subtree={"centermb": centermb, "selectedmb": self.selectednode.mb, "maxdepth": maxdepth, 
                         "bestmoves_and_score_by_board": self.bestmoves_and_score_by_board})
    self.selectednode = self.mbtree.selectednode
    self.mbtree.draw_subtree(centernode=self.mbtree.centernode, selectednode=self.selectednode,
                            show_bestmove=True, show_score=self.show_score,
                            ax=self.ax, maxdepth=maxdepth, size=self.size)
    
    disabled = self.selectednode.parent is None
    self.set_button_status(self.left_button, disabled=disabled)
    disabled = self.selectednode.depth >= 6 or len(self.selectednode.children) == 0
    self.set_button_status(self.right_button, disabled=disabled)
    disabled = self.selectednode.parent is None or self.selectednode.parent.children.index(self.selectednode) == 0
    self.set_button_status(self.up_button, disabled=disabled)
    disabled = self.selectednode.parent is None or self.selectednode.parent.children[-1] is self.selectednode
    self.set_button_status(self.down_button, disabled=disabled)
    
Mbtree_GUI.update_gui = update_gui

In [10]:
Mbtree_GUI()

In [11]:
def __init__(self, mb, parent=None, depth=0, bestmoves_and_score_by_board=None):
    self.id = Node.count
    Node.count += 1
    self.mb = mb
    self.parent = parent
    self.depth = depth
    self.children = []
    self.children_by_move = {}   
    if bestmoves_and_score_by_board is not None:
        bestmoves_and_score = bestmoves_and_score_by_board[self.mb.board_to_str()]
        self.bestmoves = bestmoves_and_score["bestmoves"]
        self.score = bestmoves_and_score["score"]   
        self.playout_prob = bestmoves_and_score["playout_prob"]
        
Node.__init__ = __init__

In [12]:
from marubatsu import Marubatsu_GUI
from tree import Rect

def draw_node(self, ax=None, maxdepth=None, emphasize=False, darkness=0, 
              show_score=True, size=0.25, lw=0.8, dx=0, dy=0):
    width = 8
    if ax is None:
        height = len(self.children) * 4
        fig, ax = plt.subplots(figsize=(width * size, height * size))
        ax.set_xlim(0, width)
        ax.set_ylim(0, height)   
        ax.invert_yaxis()
        ax.axis("off")
        for childnode in self.children:
            childnode.height = 4
        self.height = height         
        
    # 自分自身のノードを真ん中の位置になるように (dx, dy) からずらして描画する
    y = dy + (self.height - 3) / 2
    bc = "red" if emphasize else None
    Marubatsu_GUI.draw_board(ax, self.mb, show_result=True, 
                             score=getattr(self, "score", None), bc=bc, darkness=darkness, lw=lw, dx=dx, dy=y)
    if hasattr(self, "playout_prob"):
        if show_score in [self.mb.CIRCLE_STR, self.mb.CROSS_STR, self.mb.DRAW]:
            ax.text(dx, y - 0.1, f"{self.playout_prob[show_score]:.3f}", fontsize=70*size)
    if hasattr(self, "score") and show_score in [True, "score"]:
        ax.text(dx, y - 0.1, self.score, fontsize=70*size)
    rect = Rect(dx, y, 3, 3)
    # 子ノードが存在する場合に、エッジの線と子ノードを描画する
    if len(self.children) > 0:
        if maxdepth != self.depth:   
            ax.plot([dx + 3.5, dx + 4], [y + 1.5, y + 1.5], c="k", lw=lw)
            prevy = None
            for childnode in self.children:
                childnodey = dy + (childnode.height - 3) / 2
                if maxdepth is None:
                    Marubatsu_GUI.draw_board(ax, childnode.mb, show_result=True,
                                            score=getattr(childnode, "score", None), dx=dx+5, dy=childnodey, lw=lw)
                edgey = childnodey + 1.5
                ax.plot([dx + 4 , dx + 4.5], [edgey, edgey], c="k", lw=lw)
                if prevy is not None:
                    ax.plot([dx + 4 , dx + 4], [prevy, edgey], c="k", lw=lw)
                prevy = edgey
                dy += childnode.height
        else:
            ax.plot([dx + 3.5, dx + 4.5], [y + 1.5, y + 1.5], c="k", lw=lw)
            
    return rect

Node.draw_node = draw_node

In [13]:
Mbtree_GUI()

In [14]:
from marubatsu import Marubatsu
from ai import ai14s

mb = Marubatsu()
mb.cmove(1, 1)
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])

Turn x
...
.O.
...

[(0, 0), (0, 2), (2, 0), (2, 2)]


In [15]:
mb.cmove(0, 0)
mb.cmove(0, 2)
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])

Turn x
x..
.o.
O..

[(2, 0)]


In [16]:
mb.cmove(2, 0)
mb.cmove(1, 0)
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])

Turn x
xOx
.o.
o..

[(1, 2)]


In [17]:
mb.cmove(1, 2)
mb.cmove(0, 1)
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])
mb.unmove()
mb.cmove(2, 1)
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])

Turn x
xox
Oo.
ox.

[(2, 1)]
Turn x
xox
.oO
ox.

[(0, 1)]


In [18]:
mb = Marubatsu()
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])

Turn o
...
...
...

[(1, 1)]


In [19]:
mb.cmove(1, 1)
mb.cmove(0, 0)
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])

Turn o
X..
.o.
...

[(0, 2), (2, 0)]


In [20]:
mb.cmove(0, 2)
mb.cmove(2, 0)
print(mb)
candidate = ai14s(mb, analyze=True)["candidate"]
print([mb.board.move_to_xy(move) for move in candidate])

Turn o
x.X
.o.
o..

[(1, 0)]
